# Lab 4: Informed heuristic search on the Romania map

## Learning goals
By the end of this lab, students can:
- define a heuristic function h(n);
- run greedy best-first search, A* search, uniform-cost search, and weighted A*;
- explain g(n), h(n), and f(n);
- see why greedy search can be fast but not optimal;
- see how A* uses f(n) = g(n) + h(n).

Source note: The notes in this notebook are paraphrased from Chapter 3, Solving Problems by Searching, in the uploaded course document. No outside notes are used.


## Chapter 3 notes for this lab

Informed search uses problem-specific hints about closeness to a goal. A heuristic function h(n) estimates the cheapest remaining cost from node n to a goal. In the Romania route problem, the straight-line distance to Bucharest is used as a heuristic.

Greedy best-first search chooses the node with the smallest h(n), so f(n) = h(n). It often moves toward the goal quickly, but it can return a nonoptimal route.

A* chooses the node with the smallest f(n) = g(n) + h(n), combining the cost already paid with the estimated remaining cost. A* is complete and cost-optimal when the heuristic is admissible. A consistent heuristic also satisfies a triangle-inequality condition across each action.

Weighted A* uses f(n) = g(n) + W h(n), with W > 1. It can reduce search effort by focusing more strongly on the goal, while sacrificing guaranteed optimality.


In [ ]:
# Run this cell first.
# These notebooks use only local Python code plus matplotlib and ipywidgets.
# If a student machine is missing a package, uncomment and run the next line once:
# %pip install matplotlib ipywidgets

import math
import heapq
import itertools
from collections import deque
from html import escape

import matplotlib.pyplot as plt

# HTML display works even if ipywidgets is not installed.
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install with: pip install ipywidgets")


In [ ]:
ROMANIA_GRAPH = {
    "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind": {"Arad": 75, "Oradea": 71},
    "Oradea": {"Zerind": 71, "Sibiu": 151},
    "Sibiu": {"Arad": 140, "Oradea": 151, "Fagaras": 99, "Rimnicu Vilcea": 80},
    "Timisoara": {"Arad": 118, "Lugoj": 111},
    "Lugoj": {"Timisoara": 111, "Mehadia": 70},
    "Mehadia": {"Lugoj": 70, "Drobeta": 75},
    "Drobeta": {"Mehadia": 75, "Craiova": 120},
    "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
    "Rimnicu Vilcea": {"Sibiu": 80, "Craiova": 146, "Pitesti": 97},
    "Fagaras": {"Sibiu": 99, "Bucharest": 211},
    "Pitesti": {"Rimnicu Vilcea": 97, "Craiova": 138, "Bucharest": 101},
    "Bucharest": {"Fagaras": 211, "Pitesti": 101, "Giurgiu": 90, "Urziceni": 85},
    "Giurgiu": {"Bucharest": 90},
    "Urziceni": {"Bucharest": 85, "Hirsova": 98, "Vaslui": 142},
    "Hirsova": {"Urziceni": 98, "Eforie": 86},
    "Eforie": {"Hirsova": 86},
    "Vaslui": {"Urziceni": 142, "Iasi": 92},
    "Iasi": {"Vaslui": 92, "Neamt": 87},
    "Neamt": {"Iasi": 87},
}

# Coordinates are fixed local drawing coordinates, not GPS coordinates.
ROMANIA_POS = {
    "Arad": (91, 492), "Zerind": (108, 531), "Oradea": (123, 567),
    "Sibiu": (207, 457), "Timisoara": (94, 410), "Lugoj": (165, 379),
    "Mehadia": (168, 339), "Drobeta": (165, 299), "Craiova": (253, 288),
    "Rimnicu Vilcea": (233, 410), "Fagaras": (305, 449), "Pitesti": (320, 368),
    "Bucharest": (400, 327), "Giurgiu": (375, 270), "Urziceni": (456, 350),
    "Hirsova": (534, 350), "Eforie": (562, 293), "Vaslui": (509, 444),
    "Iasi": (473, 506), "Neamt": (406, 537),
}

# Straight-line distance estimates to Bucharest, used only when Bucharest is the goal.
H_SLD_BUCHAREST = {
    "Arad": 366, "Bucharest": 0, "Craiova": 160, "Drobeta": 242, "Eforie": 161,
    "Fagaras": 176, "Giurgiu": 77, "Hirsova": 151, "Iasi": 226, "Lugoj": 244,
    "Mehadia": 241, "Neamt": 234, "Oradea": 380, "Pitesti": 100,
    "Rimnicu Vilcea": 193, "Sibiu": 253, "Timisoara": 329,
    "Urziceni": 80, "Vaslui": 199, "Zerind": 374,
}

def sld_to_bucharest(city):
    return H_SLD_BUCHAREST.get(city, 0)

def edge_list(graph):
    seen = set()
    edges = []
    for a, nbrs in graph.items():
        for b, cost in nbrs.items():
            key = tuple(sorted((a, b)))
            if key not in seen:
                seen.add(key)
                edges.append((a, b, cost))
    return edges


In [ ]:
_NODE_COUNTER = itertools.count()

class Node:
    """Search-tree node: state, parent, action, path cost g(n), depth, and order for tie breaking."""
    def __init__(self, state, parent=None, action=None, path_cost=0, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = depth
        self.order = next(_NODE_COUNTER)

    def __repr__(self):
        return f"Node(state={self.state!r}, g={self.path_cost}, depth={self.depth})"

def reset_node_counter():
    global _NODE_COUNTER
    _NODE_COUNTER = itertools.count()

def path_nodes(node):
    nodes = []
    while node is not None:
        nodes.append(node)
        node = node.parent
    return list(reversed(nodes))

def path_states(node):
    return [n.state for n in path_nodes(node)]

def path_actions(node):
    return [n.action for n in path_nodes(node)[1:]]

def path_cost_of_states(graph, states):
    total = 0
    for a, b in zip(states, states[1:]):
        total += graph[a][b]
    return total

def states_to_edges(states):
    return {tuple(sorted((a, b))) for a, b in zip(states, states[1:])}

def make_html_table(headers, rows, max_rows=15):
    rows = rows[:max_rows]
    html = "<table style='border-collapse:collapse; font-family:monospace;'>"
    html += "<tr>" + "".join(f"<th style='border:1px solid #999; padding:4px 8px;'>{escape(str(h))}</th>" for h in headers) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(f"<td style='border:1px solid #999; padding:4px 8px;'>{escape(str(v))}</td>" for v in row) + "</tr>"
    html += "</table>"
    return HTML(html)

def draw_romania_map(step=None, title="Romania search map", heuristic=None, f_func=None):
    """Draw a step of search on the Romania state-space graph."""
    step = step or {}
    expanded = set(step.get("expanded", []))
    expanded_b = set(step.get("expanded_b", []))
    generated = set(step.get("generated", []))
    frontier_nodes = step.get("frontier_nodes", [])
    frontier = {n.state if hasattr(n, "state") else n for n in frontier_nodes}
    frontier_b_nodes = step.get("frontier_b_nodes", [])
    frontier_b = {n.state if hasattr(n, "state") else n for n in frontier_b_nodes}
    current = step.get("current", None)
    current_state = current.state if hasattr(current, "state") else current
    current_b = step.get("current_b", None)
    current_b_state = current_b.state if hasattr(current_b, "state") else current_b

    solution_path = step.get("solution_path", []) or []
    current_path = path_states(current) if hasattr(current, "parent") else []
    solution_edges = states_to_edges(solution_path)
    current_edges = states_to_edges(current_path)

    fig, ax = plt.subplots(figsize=(12, 7))

    # Draw edges first.
    for a, b, cost in edge_list(ROMANIA_GRAPH):
        x1, y1 = ROMANIA_POS[a]
        x2, y2 = ROMANIA_POS[b]
        edge_key = tuple(sorted((a, b)))
        if edge_key in solution_edges:
            ax.plot([x1, x2], [y1, y2], linewidth=5, color="#2ca02c", zorder=1)
        elif edge_key in current_edges:
            ax.plot([x1, x2], [y1, y2], linewidth=4, color="#ff7f0e", zorder=1)
        else:
            ax.plot([x1, x2], [y1, y2], linewidth=1, color="#999999", zorder=0)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(cost), fontsize=8, color="#444444")

    # Draw nodes.
    for city, (x, y) in ROMANIA_POS.items():
        color = "#f7f7f7"
        size = 320
        edgecolor = "#333333"
        linewidth = 1.0
        if city in expanded:
            color = "#d62728"
            size = 410
        if city in expanded_b:
            color = "#9467bd"
            size = 410
        if city in frontier:
            color = "#1f77b4"
            size = 450
        if city in frontier_b:
            color = "#17becf"
            size = 450
        if city in generated:
            color = "#ffbb78"
            size = 470
        if city == current_state or city == current_b_state:
            color = "#ff7f0e"
            size = 560
            linewidth = 2.5
        if city in solution_path:
            edgecolor = "#2ca02c"
            linewidth = 3.0
        ax.scatter([x], [y], s=size, c=color, edgecolors=edgecolor, linewidths=linewidth, zorder=3)
        label = city
        if heuristic is not None:
            label += f"\nh={heuristic(city)}"
        ax.text(x, y - 18, label, ha="center", va="top", fontsize=8)

    legend_lines = [
        "Gray edge: road with action cost",
        "Blue: frontier; red: expanded; orange: current/generated",
        "Green outline/path: solution path when found",
    ]
    if step.get("direction"):
        legend_lines.append(f"Bidirectional step direction: {step['direction']}")
    ax.text(0.01, 0.01, "\n".join(legend_lines), transform=ax.transAxes, fontsize=9,
            bbox=dict(facecolor="white", alpha=0.9, edgecolor="#cccccc"))
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

def step_summary_html(step, heuristic=None, f_func=None, max_rows=12):
    current = step.get("current")
    current_state = current.state if hasattr(current, "state") else current
    msg = step.get("message", "")
    path = path_states(current) if hasattr(current, "parent") else []
    text = f"<b>Step {step.get('step', 0)}.</b> {escape(msg)}<br>"
    if current_state is not None:
        text += f"<b>Current:</b> {escape(str(current_state))}<br>"
    if path:
        text += f"<b>Current path:</b> {' -> '.join(escape(str(s)) for s in path)}"
        if hasattr(current, "path_cost"):
            text += f" | g(n)={current.path_cost}"
        text += "<br>"

    rows = []
    for n in step.get("frontier_nodes", [])[:max_rows]:
        h = heuristic(n.state) if heuristic else 0
        f = f_func(n) if f_func else ""
        rows.append([n.state, n.depth, n.path_cost, h, f])
    table = ""
    if rows:
        table = make_html_table(["state", "depth", "g(n)", "h(n)", "f(n)"], rows, max_rows=max_rows).data
    return HTML(text + table)

def make_stepper(steps, draw_func, table_func=None, title="Step-through viewer"):
    if not steps:
        print("No steps to show.")
        return

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available, so only the final step is shown.")
        draw_func(steps[-1])
        if table_func:
            display(table_func(steps[-1]))
        return

    slider = widgets.IntSlider(value=0, min=0, max=len(steps) - 1, step=1, description="Step")
    previous_button = widgets.Button(description="Previous")
    next_button = widgets.Button(description="Next")
    play = widgets.Play(value=0, min=0, max=len(steps) - 1, step=1, interval=1200, description="Play")
    widgets.jslink((play, "value"), (slider, "value"))
    out = widgets.Output()

    def render(*args):
        with out:
            clear_output(wait=True)
            step = steps[slider.value]
            draw_func(step)
            if table_func:
                display(table_func(step))

    def go_previous(_):
        slider.value = max(slider.min, slider.value - 1)

    def go_next(_):
        slider.value = min(slider.max, slider.value + 1)

    previous_button.on_click(go_previous)
    next_button.on_click(go_next)
    slider.observe(render, names="value")
    display(HTML(f"<h3>{escape(title)}</h3>"))
    display(widgets.HBox([previous_button, next_button, play, slider]))
    display(out)
    render()


In [ ]:
def best_first_search_steps(graph, start, goal, f_func, label="Best-first search"):
    """Generic graph-search version of best-first search. A new path replaces an old one if it is cheaper."""
    reset_node_counter()
    root = Node(start)
    frontier = []
    heapq.heappush(frontier, (f_func(root), root.order, root))
    reached = {start: root}
    expanded = []
    steps = []

    def clean_frontier_nodes():
        live = [item[2] for item in frontier if reached.get(item[2].state) is item[2]]
        return sorted(live, key=lambda n: (f_func(n), n.order))

    def snapshot(message, current=None, generated=None, solution_node=None):
        steps.append({
            "step": len(steps),
            "algorithm": label,
            "message": message,
            "current": current,
            "generated": generated or [],
            "expanded": list(expanded),
            "frontier_nodes": clean_frontier_nodes(),
            "solution_path": path_states(solution_node) if solution_node else None,
        })

    snapshot(f"Initialize frontier with {start}.", current=root)

    while frontier:
        _, _, node = heapq.heappop(frontier)
        if reached.get(node.state) is not node:
            continue

        snapshot(f"Pop the lowest-priority node from the frontier: {node.state}.", current=node)

        if node.state == goal:
            snapshot(f"Goal test succeeds at {goal}. Return the solution node.", current=node, solution_node=node)
            return node, steps

        expanded.append(node.state)
        generated = []
        for nbr, cost in sorted(graph[node.state].items()):
            child = Node(nbr, parent=node, action=f"Go to {nbr}", path_cost=node.path_cost + cost, depth=node.depth + 1)
            if nbr not in reached or child.path_cost < reached[nbr].path_cost:
                reached[nbr] = child
                heapq.heappush(frontier, (f_func(child), child.order, child))
                generated.append(nbr)

        if generated:
            snapshot(f"Expand {node.state}; add or improve: {', '.join(generated)}.", current=node, generated=generated)
        else:
            snapshot(f"Expand {node.state}; no new cheaper states are added.", current=node, generated=[])

    snapshot("Frontier is empty. Search fails.")
    return None, steps

def breadth_first_search_steps(graph, start, goal):
    reset_node_counter()
    root = Node(start)
    frontier = deque([root])
    reached = {start: root}
    expanded = []
    steps = []

    def frontier_nodes():
        return list(frontier)

    def snapshot(message, current=None, generated=None, solution_node=None):
        steps.append({
            "step": len(steps),
            "algorithm": "Breadth-first search",
            "message": message,
            "current": current,
            "generated": generated or [],
            "expanded": list(expanded),
            "frontier_nodes": frontier_nodes(),
            "solution_path": path_states(solution_node) if solution_node else None,
        })

    snapshot(f"Initialize FIFO frontier with {start}.", current=root)
    if start == goal:
        snapshot("The initial state is already the goal.", current=root, solution_node=root)
        return root, steps

    while frontier:
        node = frontier.popleft()
        snapshot(f"Pop the oldest node: {node.state}.", current=node)
        expanded.append(node.state)
        generated = []
        for nbr, cost in sorted(graph[node.state].items()):
            if nbr not in reached:
                child = Node(nbr, parent=node, action=f"Go to {nbr}", path_cost=node.path_cost + cost, depth=node.depth + 1)
                reached[nbr] = child
                generated.append(nbr)
                if nbr == goal:
                    snapshot(f"Early goal test: generated {goal}, so BFS returns immediately.", current=child,
                             generated=generated, solution_node=child)
                    return child, steps
                frontier.append(child)
        snapshot(f"Expand {node.state}; append new nodes to the back of the FIFO queue.", current=node, generated=generated)

    snapshot("Frontier is empty. Search fails.")
    return None, steps

def depth_first_search_steps(graph, start, goal, depth_limit=None, cycle_check=True, label="Depth-first search"):
    reset_node_counter()
    root = Node(start)
    stack = [root]
    expanded = []
    steps = []

    def snapshot(message, current=None, generated=None, solution_node=None):
        steps.append({
            "step": len(steps),
            "algorithm": label,
            "message": message,
            "current": current,
            "generated": generated or [],
            "expanded": list(expanded),
            "frontier_nodes": list(reversed(stack)),
            "solution_path": path_states(solution_node) if solution_node else None,
        })

    snapshot(f"Initialize LIFO stack with {start}.", current=root)

    while stack:
        node = stack.pop()
        snapshot(f"Pop the newest node: {node.state}.", current=node)

        if node.state == goal:
            snapshot(f"Goal test succeeds at {goal}.", current=node, solution_node=node)
            return node, steps

        if depth_limit is not None and node.depth >= depth_limit:
            snapshot(f"Depth limit {depth_limit} reached at {node.state}; treat it as having no successors.", current=node)
            continue

        expanded.append(node.state)
        generated = []
        current_path = set(path_states(node))
        # Reverse sorted order so the alphabetically first successor is popped first.
        for nbr, cost in sorted(graph[node.state].items(), reverse=True):
            if cycle_check and nbr in current_path:
                continue
            child = Node(nbr, parent=node, action=f"Go to {nbr}", path_cost=node.path_cost + cost, depth=node.depth + 1)
            stack.append(child)
            generated.append(nbr)
        snapshot(f"Expand {node.state}; push successors on the LIFO stack.", current=node, generated=list(reversed(generated)))

    snapshot("Frontier is empty. Search fails or cutoff prevented reaching a goal.")
    return None, steps

def iterative_deepening_steps(graph, start, goal, max_depth=12):
    all_steps = []
    for limit in range(max_depth + 1):
        node, steps = depth_first_search_steps(
            graph, start, goal, depth_limit=limit, cycle_check=True,
            label=f"Iterative deepening DFS, limit={limit}"
        )
        for s in steps:
            s = dict(s)
            s["step"] = len(all_steps)
            s["message"] = f"Depth limit {limit}: {s['message']}"
            all_steps.append(s)
        if node is not None:
            return node, all_steps
    return None, all_steps

def bidirectional_bfs_steps(graph, start, goal):
    reset_node_counter()
    if start == goal:
        root = Node(start)
        return root, [{
            "step": 0, "algorithm": "Bidirectional BFS",
            "message": "Start and goal are the same.",
            "current": root, "expanded": [], "frontier_nodes": [],
            "current_b": None, "expanded_b": [], "frontier_b_nodes": [],
            "solution_path": [start], "direction": "none"
        }]

    fw_root = Node(start)
    bw_root = Node(goal)
    fw_q = deque([fw_root])
    bw_q = deque([bw_root])
    fw_reached = {start: fw_root}
    bw_reached = {goal: bw_root}
    fw_expanded = []
    bw_expanded = []
    steps = []

    def combine(meeting):
        fw_path = path_states(fw_reached[meeting])
        bw_path = path_states(bw_reached[meeting])  # goal -> ... -> meeting
        return fw_path + list(reversed(bw_path))[1:]  # start -> ... -> meeting -> ... -> goal

    def snapshot(message, direction, current=None, current_b=None, generated=None, solution_path=None):
        steps.append({
            "step": len(steps),
            "algorithm": "Bidirectional BFS",
            "message": message,
            "direction": direction,
            "current": current,
            "current_b": current_b,
            "generated": generated or [],
            "expanded": list(fw_expanded),
            "expanded_b": list(bw_expanded),
            "frontier_nodes": list(fw_q),
            "frontier_b_nodes": list(bw_q),
            "solution_path": solution_path,
        })

    snapshot(f"Initialize forward frontier at {start} and backward frontier at {goal}.", "initialize", fw_root, bw_root)

    while fw_q and bw_q:
        if len(fw_q) <= len(bw_q):
            node = fw_q.popleft()
            fw_expanded.append(node.state)
            generated = []
            for nbr, cost in sorted(graph[node.state].items()):
                if nbr not in fw_reached:
                    child = Node(nbr, parent=node, action=f"Forward to {nbr}", path_cost=node.path_cost + cost, depth=node.depth + 1)
                    fw_reached[nbr] = child
                    fw_q.append(child)
                    generated.append(nbr)
                    if nbr in bw_reached:
                        solution_path = combine(nbr)
                        snapshot(f"Forward search reaches {nbr}, already reached by backward search. Frontiers meet.", "forward", child, None, generated, solution_path)
                        return child, steps
            snapshot(f"Expand forward node {node.state}.", "forward", node, None, generated)
        else:
            node = bw_q.popleft()
            bw_expanded.append(node.state)
            generated = []
            for nbr, cost in sorted(graph[node.state].items()):
                if nbr not in bw_reached:
                    child = Node(nbr, parent=node, action=f"Backward to {nbr}", path_cost=node.path_cost + cost, depth=node.depth + 1)
                    bw_reached[nbr] = child
                    bw_q.append(child)
                    generated.append(nbr)
                    if nbr in fw_reached:
                        solution_path = combine(nbr)
                        snapshot(f"Backward search reaches {nbr}, already reached by forward search. Frontiers meet.", "backward", None, child, generated, solution_path)
                        return child, steps
            snapshot(f"Expand backward node {node.state}.", "backward", None, node, generated)

    snapshot("One frontier is empty. Search fails.", "failure")
    return None, steps


## Code: informed search algorithms


In [ ]:
def greedy_f(node):
    return sld_to_bucharest(node.state)

def astar_f(node):
    return node.path_cost + sld_to_bucharest(node.state)

def weighted_astar_f(weight):
    return lambda node: node.path_cost + weight * sld_to_bucharest(node.state)

def run_informed(name, start="Arad", weight=1.5):
    goal = "Bucharest"
    if name == "Greedy best-first search":
        return best_first_search_steps(ROMANIA_GRAPH, start, goal, greedy_f, label="Greedy best-first search")
    if name == "A* search":
        return best_first_search_steps(ROMANIA_GRAPH, start, goal, astar_f, label="A* search")
    if name == "Weighted A* search":
        return best_first_search_steps(ROMANIA_GRAPH, start, goal, weighted_astar_f(weight), label=f"Weighted A* search, W={weight}")
    if name == "Uniform-cost search":
        return best_first_search_steps(ROMANIA_GRAPH, start, goal, lambda n: n.path_cost, label="Uniform-cost search")
    raise ValueError(name)

def summarize_informed(name, node, steps, f):
    path = steps[-1].get("solution_path") or (path_states(node) if node else [])
    return {
        "algorithm": name,
        "path": " -> ".join(path),
        "path_cost": path_cost_of_states(ROMANIA_GRAPH, path) if len(path) > 1 else 0,
        "recorded_steps": len(steps),
        "expanded_count": len(steps[-1].get("expanded", [])) if steps else 0,
    }

informed_names = ["Greedy best-first search", "A* search", "Weighted A* search", "Uniform-cost search"]
rows = []
for name in informed_names:
    node, steps = run_informed(name, start="Arad", weight=1.5)
    summary = summarize_informed(name, node, steps, None)
    rows.append([summary["algorithm"], summary["path"], summary["path_cost"], summary["expanded_count"], summary["recorded_steps"]])

make_html_table(["algorithm", "path", "path cost", "expanded count", "recorded steps"], rows, max_rows=10)


## GUI: choose greedy, A*, weighted A*, or uniform-cost search


In [ ]:
def informed_table(step, active_name, weight=1.5):
    if active_name == "Greedy best-first search":
        f_func = greedy_f
    elif active_name == "A* search":
        f_func = astar_f
    elif active_name == "Weighted A* search":
        f_func = weighted_astar_f(weight)
    else:
        f_func = lambda n: n.path_cost
    return step_summary_html(step, heuristic=sld_to_bucharest, f_func=f_func)

def informed_gui():
    if not WIDGETS_AVAILABLE:
        node, steps = run_informed("A* search")
        make_stepper(
            steps,
            draw_func=lambda step: draw_romania_map(step, title="A* search", heuristic=sld_to_bucharest),
            table_func=lambda step: informed_table(step, "A* search"),
            title="A* search GUI map"
        )
        return

    algorithm_dropdown = widgets.Dropdown(options=informed_names, value="A* search", description="Algorithm")
    start_dropdown = widgets.Dropdown(options=sorted(ROMANIA_GRAPH.keys()), value="Arad", description="Start")
    weight_slider = widgets.FloatSlider(value=1.5, min=1.0, max=3.0, step=0.1, description="W")
    run_button = widgets.Button(description="Run search")
    out = widgets.Output()

    def on_run(_=None):
        with out:
            clear_output(wait=True)
            node, steps = run_informed(algorithm_dropdown.value, start=start_dropdown.value, weight=weight_slider.value)
            path = steps[-1].get("solution_path") or (path_states(node) if node else [])
            print("Goal is fixed to Bucharest because this lab uses hSLD to Bucharest.")
            if path:
                print("Solution:", " -> ".join(path))
                print("Path cost:", path_cost_of_states(ROMANIA_GRAPH, path))
            print("h(start) =", sld_to_bucharest(start_dropdown.value))
            make_stepper(
                steps,
                draw_func=lambda step: draw_romania_map(
                    step,
                    title=f"{algorithm_dropdown.value}: goal Bucharest",
                    heuristic=sld_to_bucharest
                ),
                table_func=lambda step: informed_table(step, algorithm_dropdown.value, weight_slider.value),
                title=f"{algorithm_dropdown.value} GUI map"
            )

    run_button.on_click(on_run)
    display(widgets.VBox([widgets.HBox([algorithm_dropdown, start_dropdown]), weight_slider, run_button, out]))
    on_run()

informed_gui()


## Check heuristic consistency for hSLD on the Romania roads


In [ ]:
def check_consistency(graph, h):
    violations = []
    for a, nbrs in graph.items():
        for b, cost in nbrs.items():
            if h(a) > cost + h(b):
                violations.append((a, b, h(a), cost, h(b)))
    return violations

violations = check_consistency(ROMANIA_GRAPH, sld_to_bucharest)
if violations:
    print("Heuristic consistency violations found:")
    for row in violations:
        print(row)
else:
    print("No violations: hSLD is consistent on these Romania edges.")

# Show the triangle inequality calculation for a few edges.
sample_edges = [("Arad", "Sibiu"), ("Sibiu", "Rimnicu Vilcea"), ("Pitesti", "Bucharest")]
for a, b in sample_edges:
    print(f"h({a}) <= c({a},{b}) + h({b})  ->  {sld_to_bucharest(a)} <= {ROMANIA_GRAPH[a][b]} + {sld_to_bucharest(b)}")


## Student practice


In [ ]:
# Practice: Greedy search and A* may return different routes.
for name in ["Greedy best-first search", "A* search"]:
    node, steps = run_informed(name, start="Arad")
    path = steps[-1].get("solution_path") or path_states(node)
    print(name)
    print("  path:", " -> ".join(path))
    print("  path cost:", path_cost_of_states(ROMANIA_GRAPH, path))
    print("  expanded:", len(steps[-1].get("expanded", [])))
